In [1]:
import pandas as pd
import numpy as np

print("[*] 120k dengeli altın veri seti yükleniyor...")
df = pd.read_csv('C:/labs/data/islenmis/gold_dataset_120k.csv' , low_memory=False)

print('Veri seti başarıyla yüklendi')

[*] 120k dengeli altın veri seti yükleniyor...
Veri seti başarıyla yüklendi


In [3]:
print("[*] Özellik mühendisliği adımları başlatıldı...")

# 1. Paket Boyutu (frame.len) için Mantıksal Gruplama (Binning)
def boyut_kategorisi(boyut):
    if boyut <= 64:
        return 0   # Çok Küçük Kontrol/Flood Paketleri
    elif boyut <= 128:
        return 1   # Küçük Paketler
    elif boyut <= 512:
        return 2   # Orta Boy Paketler (Örn: HTTP istekleri)
    elif boyut <= 1024:
        return 3   # Büyük Paketler
    else:
        return 4   # Maksimum Boyuta Yakın Paketler (Veri transferi / MTU)

df['packet_size_category'] = df['frame.len'].apply(boyut_kategorisi)
print("[+] 'frame.len' özelliği 'packet_size_category' olarak mantıksal gruplara ayrıldı.")

# 2. String Olabilecek Ağ Özelliklerini Temizleme ve Sayısala Çevirme
kolonlar_to_numeric = ['tcp.flags', 'tcp.window_size_value', 'tcp.srcport', 'tcp.dstport', 'udp.srcport', 'udp.dstport', 'ip.proto']

for col in kolonlar_to_numeric:
    if col in df.columns:
        # Eğer kolon nesne (string) tipindeyse tırnakları temizle
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace('"', '').str.strip()
            df[col] = df[col].apply(lambda x: int(x, 16) if str(x).startswith('0x') else pd.to_numeric(x, errors='coerce'))
        
        # MODERN YAKLAŞIM: inplace=True kullanmak yerine doğrudan kolona eşitliyoruz
        df[col] = df[col].fillna(0)
        df[col] = df[col].astype(int)

print("[+] Tüm ağ özellikleri (Portlar, Flagler, Protokoller) başarıyla tam sayıya (int) dönüştürüldü.")

[*] Özellik mühendisliği adımları başlatıldı...
[+] 'frame.len' özelliği 'packet_size_category' olarak mantıksal gruplara ayrıldı.
[+] Tüm ağ özellikleri (Portlar, Flagler, Protokoller) başarıyla tam sayıya (int) dönüştürüldü.


In [4]:
tehlikeli_kolonlar = ['frame.time_relative' , 'frame.len']

df_final = df.drop(columns=[col for col in tehlikeli_kolonlar if col in df.columns])

print("\n=== EĞİTİME HAZIR HALE GELEN YENİ SÜTUNLAR ===")
print(df_final.columns.tolist())


=== EĞİTİME HAZIR HALE GELEN YENİ SÜTUNLAR ===
['ip.proto', 'tcp.srcport', 'tcp.dstport', 'udp.srcport', 'udp.dstport', 'tcp.flags', 'tcp.window_size_value', 'label', 'packet_size_category']


In [5]:
output_final_path = 'C:/labs/data/islenmis/ready_to_train_120k.csv'
df_final.to_csv(output_final_path , index=False)

print(f"[+] Tebrikler! Veri önişleme başarıyla tamamlandı.")
print(f"[+] Eğitime hazır veri seti şuraya kaydedildi: {output_final_path}")

[+] Tebrikler! Veri önişleme başarıyla tamamlandı.
[+] Eğitime hazır veri seti şuraya kaydedildi: C:/labs/data/islenmis/ready_to_train_120k.csv


In [6]:
import pandas as pd
import numpy as np

# 1. Mevcut veri setini yükle
df_engineering = pd.read_csv("C:/labs/data/islenmis/ready_to_train_120k.csv")

print(f"[*] Orijinal Satır Sayısı: {len(df_engineering)}")

# 2. ICMP ve Kayıt Hatası Olan Protokolleri Uçur (Sadece TCP=6 ve UDP=17 Kalsın)
# Böylece flag'i ve portu 0 olan sahte gürültü paketlerinden kurtuluyoruz
df_engineering = df_engineering[df_engineering['ip.proto'].isin([6, 17])].reset_index(drop=True)
print(f"[+] ICMP ve Gürültü Paketleri Temizlendi. Kalan Satır: {len(df_engineering)}")

# 3. Port Numaraları İçin Siber Güvenlik Odaklı Gruplama (Port Binning)
# Portları büyüklük ilişkisinden kurtarıp kategorik anlam yüklüyoruz
def port_kategorize(port):
    if port == 0:
        return 0  # Portsuz/Boş paket (UDP veya hatalı)
    elif port in [21, 22, 23, 25, 53, 80, 110, 443, 445, 3389]:
        return 1  # Kritik Kurumsal Portlar (SSH, HTTP, RDP, FTP vb.)
    elif port < 1024:
        return 2  # Diğer Sistem Portları (Well-Known Ports)
    elif port < 49152:
        return 3  # Kayıtlı Portlar (Registered Ports)
    else:
        return 4  # Dinamik/Özel Portlar (Dynamic/Private Ports)

# Hem kaynak hem hedef portları siber güvenlik mantığına göre sadeleştiriyoruz
df_engineering['src_port_category'] = df_engineering['tcp.srcport'].copy().apply(port_kategorize)
df_engineering['dst_port_category'] = df_engineering['tcp.dstport'].copy().apply(port_kategorize)

# Eğer UDP portları doluysa onları da buraya yedekle
df_engineering.loc[df_engineering['ip.proto'] == 17, 'src_port_category'] = df_engineering['udp.srcport'].apply(port_kategorize)
df_engineering.loc[df_engineering['ip.proto'] == 17, 'dst_port_category'] = df_engineering['udp.dstport'].apply(port_kategorize)

# 4. Eski Ham Port Kolonlarını Sil (Artık kategorilerimiz var, yapay zeka ezberleyemez)
eski_portlar = ['tcp.srcport', 'tcp.dstport', 'udp.srcport', 'udp.dstport']
df_final_mhendislik = df_engineering.drop(columns=eski_portlar)

# 5. Mükemmel Hale Gelen Veri Setini Kaydet
final_output_path = "C:/labs/data/islenmis/perfect_dataset_for_ml.csv"
df_final_mhendislik.to_csv(final_output_path, index=False)

print("\n================ MÜHENDİSLİK OPERASYONU TAMAMLANDI ================")
print(f"Yeni Veri Seti Sütunları: {df_final_mhendislik.columns.tolist()}")
print(f"Son Satır Sayısı (Dengeli ve Temiz): {len(df_final_mhendislik)}")

[*] Orijinal Satır Sayısı: 120000
[+] ICMP ve Gürültü Paketleri Temizlendi. Kalan Satır: 97277

================ MÜHENDİSLİK OPERASYONU TAMAMLANDI ================
Yeni Veri Seti Sütunları: ['ip.proto', 'tcp.flags', 'tcp.window_size_value', 'label', 'packet_size_category', 'src_port_category', 'dst_port_category']
Son Satır Sayısı (Dengeli ve Temiz): 97277
